In [52]:
import pandas as pd
import plotly.express as px
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster,cophenet
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

### Estandarización y selección de columnas para cluster de productos

In [41]:
#Leyendo el dataset
resumen_productos = pd.read_csv('../data_clean/resumen_productos.csv')
resumen_productos.head(5)

,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock,Unidades_Vendidas,Venta_Total,Transacciones,%_Unidades,%_Venta,Cancelada,Completa,Pendiente
0,6,Asado,Carnicería,28.56,5137,299,8539.44,81,2.863710,8.282421,0,71,10
1,8,Milanesa,Carnicería,16.21,3140,320,5187.20,89,3.064841,5.031076,0,76,13
2,25,Pizza congelada,Congelados,15.45,1640,332,5129.40,86,3.179772,4.975016,1,79,6
3,4,Queso rallado,Lácteos,19.23,2099,259,4980.57,77,2.480605,4.830665,1,67,9
4,3,Queso cremoso,Lácteos,17.23,3167,268,4617.64,84,2.566804,4.478659,0,66,18


In [44]:
resumen_productos['Porc_Canceladas'] = resumen_productos['Cancelada']/resumen_productos['Transacciones'] *100
resumen_productos['Porc_Completadas'] = resumen_productos['Completa']/resumen_productos['Transacciones'] *100
resumen_productos['Porc_Pendientes'] = resumen_productos['Pendiente']/resumen_productos['Transacciones'] *100


resumen_productos['Porc_Unidades'] = resumen_productos['%_Unidades']/100
resumen_productos['Porc_Ventas'] = resumen_productos['%_Venta']/100

In [46]:
resumen_productos = resumen_productos.drop(columns=['Cancelada', 'Completa', 'Pendiente', '%_Unidades', '%_Venta'])

resumen_productos.head(5)

,ID_Producto,Nombre_producto,Categoría,Precio_Unitario,Stock,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Porc_Unidades,Porc_Ventas
0,6,Asado,Carnicería,28.56,5137,299,8539.44,81,0.000000,87.654321,12.345679,0.028637,0.082824
1,8,Milanesa,Carnicería,16.21,3140,320,5187.20,89,0.000000,85.393258,14.606742,0.030648,0.050311
2,25,Pizza congelada,Congelados,15.45,1640,332,5129.40,86,1.162791,91.860465,6.976744,0.031798,0.049750
3,4,Queso rallado,Lácteos,19.23,2099,259,4980.57,77,1.298701,87.012987,11.688312,0.024806,0.048307
4,3,Queso cremoso,Lácteos,17.23,3167,268,4617.64,84,0.000000,78.571429,21.428571,0.025668,0.044787


In [48]:
#Filtrando las posibles columnas a usar
interest_cols = ['Categoría', 'Unidades_Vendidas', 'Venta_Total', 'Transacciones', 'Porc_Unidades', 'Porc_Ventas', 'Precio_Unitario', 'Porc_Canceladas', 'Porc_Completadas', 'Porc_Pendientes', 'Stock']
X_prod = resumen_productos[interest_cols]
X_prod = X_prod.rename(columns={'Categoría': 'Categoria'})
X_prod.head(5)

,Categoria,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Unidades,Porc_Ventas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Stock
0,Carnicería,299,8539.44,81,0.028637,0.082824,28.56,0.000000,87.654321,12.345679,5137
1,Carnicería,320,5187.20,89,0.030648,0.050311,16.21,0.000000,85.393258,14.606742,3140
2,Congelados,332,5129.40,86,0.031798,0.049750,15.45,1.162791,91.860465,6.976744,1640
3,Lácteos,259,4980.57,77,0.024806,0.048307,19.23,1.298701,87.012987,11.688312,2099
4,Lácteos,268,4617.64,84,0.025668,0.044787,17.23,0.000000,78.571429,21.428571,3167


In [63]:
#Aplicando OneHotEncoder a categoria 

X_prod = pd.get_dummies(X_prod, drop_first=True, dtype=int)

In [64]:
X_prod

,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Unidades,Porc_Ventas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Stock,Categoria_Carnicería,Categoria_Congelados,Categoria_Conservas,Categoria_Frutas y Verduras,Categoria_Galletitas y Snacks,Categoria_Lácteos,Categoria_Panadería
0,299.0,8539.44,81.0,0.028637,0.082824,28.56,0.000000,87.654321,12.345679,5137.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,320.0,5187.20,89.0,0.030648,0.050311,16.21,0.000000,85.393258,14.606742,3140.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
2,332.0,5129.40,86.0,0.031798,0.049750,15.45,1.162791,91.860465,6.976744,1640.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,259.0,4980.57,77.0,0.024806,0.048307,19.23,1.298701,87.012987,11.688312,2099.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,268.0,4617.64,84.0,0.025668,0.044787,17.23,0.000000,78.571429,21.428571,3167.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
5,246.0,4565.76,71.0,0.023561,0.044283,18.56,0.000000,85.915493,14.084507,4051.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,350.0,4039.00,98.0,0.033522,0.039174,11.54,0.000000,89.795918,10.204082,2455.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,277.0,3750.58,75.0,0.026530,0.036377,13.54,0.000000,85.333333,14.666667,2908.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
8,375.0,3577.50,105.0,0.035916,0.034698,9.54,0.000000,81.904762,18.095238,1688.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
9,250.0,3562.50,68.0,0.023944,0.034553,14.25,1.470588,77.941176,20.588235,2429.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
# Aplicando el StandardScaler a los datos 
scaler = StandardScaler()
X_scaled_values = scaler.fit_transform(X_prod)
X_prod_scaled = pd.DataFrame(X_scaled_values, columns=X_prod.columns)

X_prod_scaled.head(5)

,Unidades_Vendidas,Venta_Total,Transacciones,Porc_Unidades,Porc_Ventas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Porc_Pendientes,Stock,Categoria_Carnicería,Categoria_Congelados,Categoria_Conservas,Categoria_Frutas y Verduras,Categoria_Galletitas y Snacks,Categoria_Lácteos,Categoria_Panadería
0,0.589461,3.640095,0.217133,0.589461,3.640095,3.392403,-0.425004,0.766728,-0.708987,1.746380,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
1,1.100200,1.545681,1.063395,1.100200,1.545681,1.154800,-0.425004,0.279457,-0.209220,0.002711,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
2,1.392051,1.509568,0.746046,1.392051,1.509568,1.017101,0.986195,1.673173,-1.895691,-1.307005,-0.433013,2.915476,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
3,-0.383374,1.416582,-0.205998,-0.383374,1.416582,1.701971,1.151140,0.628517,-0.854286,-0.906232,-0.433013,-0.342997,-0.342997,-0.433013,-0.342997,2.569047,-0.389249
4,-0.164486,1.189831,0.534481,-0.164486,1.189831,1.339606,-0.425004,-1.190681,1.298620,0.026286,-0.433013,-0.342997,-0.342997,-0.433013,-0.342997,2.569047,-0.389249


In [69]:
correlation =  X_prod.corr()

fig = px.imshow(correlation, text_auto=".2f")
fig.update_layout(width = 1200, height = 1200)
fig.show()

Como se puede observar hay diversas variables que tienen una alta correlación entre si lo cuál no es adecuado para lograr hacer clustering, debido a ello se utilizarán únicamente las columnas: 

- Porc_Completadas
- Porc_Canceladas
- Precio_Unitario
- Stock
- Unidades_Vendidas
- Todas las categorías

In [72]:
not_used_cols = ['Venta_Total', 'Transacciones', 'Porc_Unidades', 'Porc_Ventas', 'Porc_Pendientes']

X_prod_scaled = X_prod_scaled.drop(columns=not_used_cols)

X_prod_scaled

,Unidades_Vendidas,Precio_Unitario,Porc_Canceladas,Porc_Completadas,Stock,Categoria_Carnicería,Categoria_Congelados,Categoria_Conservas,Categoria_Frutas y Verduras,Categoria_Galletitas y Snacks,Categoria_Lácteos,Categoria_Panadería
0,0.589461,3.392403,-0.425004,0.766728,1.746380,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
1,1.100200,1.154800,-0.425004,0.279457,0.002711,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
2,1.392051,1.017101,0.986195,1.673173,-1.307005,-0.433013,2.915476,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
3,-0.383374,1.701971,1.151140,0.628517,-0.906232,-0.433013,-0.342997,-0.342997,-0.433013,-0.342997,2.569047,-0.389249
4,-0.164486,1.339606,-0.425004,-1.190681,0.026286,-0.433013,-0.342997,-0.342997,-0.433013,-0.342997,2.569047,-0.389249
5,-0.699545,1.580579,-0.425004,0.392001,0.798146,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
6,1.829826,0.308678,-0.425004,1.228253,-0.595392,-0.433013,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
7,0.054402,0.671043,-0.425004,0.266543,-0.199858,-0.433013,2.915476,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
8,2.437848,-0.053687,-0.425004,-0.472331,-1.265094,-0.433013,2.915476,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
9,-0.602262,0.799682,1.359748,-1.326504,-0.618094,2.309401,-0.342997,-0.342997,-0.433013,-0.342997,-0.389249,-0.389249
